# Replication of Dadfar (2026): Self-Contained Experiment Notebook

This notebook reproduces all experiments from:

> **Replication and Remediation of "When Models Examine Themselves: Vocabulary-Activation Correspondence in Self-Referential LLM Processing"** (Dadfar, 2026)

**Self-contained**: No external repo clone required. All code is inlined.

**Phases:**
1. **Compliance test** — 2000-token screen across all prompt conditions
2. **Phase 1: Extended baseline** — N full-length runs with activation capture
3. **Phase 2: Truncation analysis** — CPU-only post-processing of Phase 1
4. **Phase 3: Control runs** — All conditions with loop-detection early termination
5. **Phase 4: Direction extraction** — Contrastive directions + topic vector comparison

**Hardware requirements:**
- A100 (40/80 GB): Llama 70B in 4-bit (~36 GiB), all smaller models
- T4 (16 GB): Llama 8B, Mistral 7B, Gemma 9B in 4-bit
- L4 (24 GB): Same as T4 + Qwen 32B

**Estimated runtime (A100, 50 baseline + 100 control runs):** ~6-10 hours

## 1. Environment Setup

In [ ]:
# Install dependencies
# autoawq provides pre-quantized model support (no on-the-fly quantization)
# bitsandbytes still needed for smaller models that don't have AWQ variants
!pip install -q transformers accelerate bitsandbytes autoawq scipy huggingface_hub

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"

# Mount Google Drive for persistent output (optional)
MOUNT_DRIVE = True  # Set False to save to /content only

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = '/content/drive/MyDrive/dadfar_replication'
else:
    OUTPUT_ROOT = '/content/dadfar_replication'

os.makedirs(OUTPUT_ROOT, exist_ok=True)
print(f"Output root: {OUTPUT_ROOT}")

In [ ]:
# HuggingFace login (required for Llama, Gemma)
from huggingface_hub import login
login()  # Paste your token when prompted

## 2. Configuration

Edit this cell to choose your model and run parameters.

In [ ]:
# ========================================================================
# CONFIGURATION
# ========================================================================

# Model profile
# Options: "llama_8b", "mistral_7b", "gemma_9b", "qwen_32b", "llama_70b"
PROFILE = "llama_70b"

# Override run counts (None = use profile defaults)
N_BASELINE_RUNS = None      # Phase 1 (default: 50 for 70B, 30 for others)
N_CONTROL_RUNS = None       # Phase 3 per condition (default: 10 for 70B, 5 for others)

# Which phases to run
RUN_COMPLIANCE = True
RUN_PHASE1 = True
RUN_PHASE2 = True
RUN_PHASE3 = True
RUN_PHASE4 = True

## 3. Core Library

All functions inlined from the project's `src/` and `scripts/` modules.
This cell defines everything needed; no external imports required.

In [ ]:
import re
import json
import sys
import time
import gc
import math
from collections import Counter
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Optional

import torch
import numpy as np
from scipy import stats
import transformers

# Detect transformers 5.x: torch_dtype was renamed to dtype
_TF_MAJOR = int(transformers.__version__.split(".")[0])
_DTYPE_KEY = "dtype" if _TF_MAJOR >= 5 else "torch_dtype"


# ====================================================================
# LOGIT SANITIZER — prevents inf/nan crash in torch.multinomial
# ====================================================================
# BnB NF4 with float16 compute can produce logit values that overflow
# float16 range (~65504), yielding inf/nan after softmax. This processor
# clamps those values before sampling. Greedy decoding (argmax) is
# unaffected; only sampled generation needs this guard.

class SanitizeLogitsProcessor:
    """Replace inf/nan logits with large negative value (effectively zero probability)."""
    def __call__(self, input_ids, scores):
        scores = torch.where(
            torch.isnan(scores) | torch.isinf(scores),
            torch.full_like(scores, -1e4),
            scores,
        )
        return scores

_LOGIT_SANITIZER = [SanitizeLogitsProcessor()]


# ====================================================================
# ACTIVATION METRICS (src/metrics/activation_metrics.py)
# ====================================================================

def compute_all_metrics(norms, vectors=None):
    """Compute all activation metrics from per-token data."""
    n = len(norms)
    if n < 2:
        return {
            "mean_norm": 0.0, "max_norm": 0.0, "norm_std": 0.0,
            "norm_kurtosis": 0.0, "autocorr_lag1": 0.0, "autocorr_lag2": 0.0,
            "spectral_power_low": 0.0, "spectral_power_mid": 0.0,
            "mean_derivative": 0.0, "max_derivative": 0.0,
            "sign_changes": 0, "sign_change_rate": 0.0,
            "mean_token_similarity": 0.0, "token_similarity_std": 0.0,
            "sparsity": 0.0, "convergence_ratio": 0.0, "n_tokens": n,
        }

    def _autocorr(norms, lag=1):
        if len(norms) <= lag:
            return 0.0
        r, _ = stats.pearsonr(norms[:-lag], norms[lag:])
        return float(r) if not np.isnan(r) else 0.0

    def _spectral_power(norms, band="low"):
        centered = norms - np.mean(norms)
        fft_vals = np.fft.rfft(centered)
        power = np.abs(fft_vals) ** 2
        n_freqs = len(power)
        if band == "low":
            lo, hi = 1, max(2, n_freqs // 20)
        elif band == "mid":
            lo = max(2, n_freqs // 20)
            hi = max(lo + 1, n_freqs // 4)
        else:
            raise ValueError(f"Unknown band: {band}")
        return float(np.sum(power[lo:hi]))

    def _convergence_ratio(norms):
        n = len(norms)
        w = max(1, n // 10)
        first = np.mean(norms[:w])
        last = np.mean(norms[-w:])
        return float(last / first) if first > 1e-10 else 0.0

    diffs = np.diff(norms)
    metrics = {
        "mean_norm": float(np.mean(norms)),
        "max_norm": float(np.max(norms)),
        "norm_std": float(np.std(norms)),
        "norm_kurtosis": float(stats.kurtosis(norms)) if n > 3 else 0.0,
        "autocorr_lag1": _autocorr(norms, 1),
        "autocorr_lag2": _autocorr(norms, 2),
        "spectral_power_low": _spectral_power(norms, "low"),
        "spectral_power_mid": _spectral_power(norms, "mid"),
        "mean_derivative": float(np.mean(np.abs(diffs))),
        "max_derivative": float(np.max(np.abs(diffs))),
        "sign_changes": 0, "sign_change_rate": 0.0,
        "mean_token_similarity": 0.0, "token_similarity_std": 0.0,
        "sparsity": 0.0,
        "convergence_ratio": _convergence_ratio(norms),
        "n_tokens": n,
    }

    if vectors is not None and len(vectors) > 1:
        # Sparsity
        metrics["sparsity"] = float(np.mean(np.abs(vectors) < 0.1))
        # Sign change rate via first PC
        n_tok, _ = vectors.shape
        if n_tok >= 3:
            centered = vectors - np.mean(vectors, axis=0, keepdims=True)
            sample = centered[:10000] if n_tok > 10000 else centered
            _, _, Vt = np.linalg.svd(sample, full_matrices=False)
            projected = centered @ Vt[0]
            signs = np.sign(projected)
            changes = int(np.sum(signs[:-1] != signs[1:]))
            metrics["sign_changes"] = changes
            metrics["sign_change_rate"] = float(changes / (n_tok - 1))
        # Token similarity
        vec_norms = np.linalg.norm(vectors, axis=-1, keepdims=True)
        vec_norms = np.maximum(vec_norms, 1e-10)
        normalized = vectors / vec_norms
        sims = np.sum(normalized[:-1] * normalized[1:], axis=-1)
        metrics["mean_token_similarity"] = float(np.mean(sims))
        metrics["token_similarity_std"] = float(np.std(sims))

    return metrics


# ====================================================================
# LOOP DETECTION (src/analysis/loop_detection.py)
# ====================================================================

@dataclass
class CycleResult:
    has_cycle: bool = False
    lock_in_obs: Optional[int] = None
    cycle_period: Optional[int] = None
    cycle_vocabulary: list = field(default_factory=list)
    exploration_vocabulary: list = field(default_factory=list)
    n_observations: int = 0
    n_unique: int = 0
    unique_ratio: float = 0.0
    observations: list = field(default_factory=list)
    state_sequence: list = field(default_factory=list)


def parse_observations(text):
    """Parse numbered observations from Pull Methodology output."""
    lines = text.strip().split('\n')
    observations = []
    current_obs = None
    current_num = 0
    for line in lines:
        line = line.strip()
        if not line:
            continue
        m = re.match(r'^(?:(?:Pull|Observation)\s+)?(\d+)[.\):](.*)', line, re.IGNORECASE)
        if m:
            num = int(m.group(1))
            obs_text = m.group(2).strip()
            if num > current_num:
                if current_obs is not None:
                    observations.append(current_obs)
                current_obs = obs_text
                current_num = num
            elif num == current_num and current_obs is not None:
                current_obs += ' ' + obs_text
        elif current_obs is not None:
            current_obs += ' ' + line
    if current_obs is not None:
        observations.append(current_obs)
    return observations


def normalize_observation(obs):
    obs = obs.lower()
    obs = re.sub(r'[^\w\s]', '', obs)
    obs = re.sub(r'\s+', ' ', obs).strip()
    return obs


def observation_to_bow(obs):
    norm = normalize_observation(obs)
    stopwords = {
        'a', 'an', 'the', 'of', 'in', 'to', 'for', 'is', 'on', 'and',
        'or', 'as', 'by', 'with', 'from', 'that', 'this', 'it', 'its',
        'are', 'was', 'be', 'been', 'being', 'have', 'has', 'had',
        'through', 'about', 'between', 'than', 'more', 'not', 'but',
    }
    return Counter(w for w in norm.split() if w not in stopwords)


def cosine_similarity_bow(a, b):
    if not a or not b:
        return 0.0
    common = set(a.keys()) & set(b.keys())
    dot = sum(a[k] * b[k] for k in common)
    norm_a = math.sqrt(sum(v * v for v in a.values()))
    norm_b = math.sqrt(sum(v * v for v in b.values()))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


def detect_cycle_exact(observations, min_period=3, max_period=50, min_confirmations=3):
    n = len(observations)
    norms = [normalize_observation(obs) for obs in observations]
    for period in range(min_period, min(max_period + 1, n // min_confirmations)):
        for start in range(n - period * min_confirmations):
            confirmed = 0
            for k in range(1, (n - start) // period):
                if norms[start + k * period] == norms[start]:
                    confirmed += 1
                else:
                    break
            if confirmed >= min_confirmations:
                all_match = True
                for offset in range(period):
                    base = norms[start + offset]
                    for k in range(1, min_confirmations + 1):
                        idx = start + offset + k * period
                        if idx >= n or norms[idx] != base:
                            all_match = False
                            break
                    if not all_match:
                        break
                if all_match:
                    return start + 1, period
    return None, None


def detect_cycle_similarity(observations, min_period=3, max_period=50,
                            threshold=0.85, min_confirmations=3):
    n = len(observations)
    bows = [observation_to_bow(obs) for obs in observations]
    for period in range(min_period, min(max_period + 1, n // min_confirmations)):
        for start in range(n - period * min_confirmations):
            all_match = True
            for offset in range(period):
                base_bow = bows[start + offset]
                for k in range(1, min_confirmations + 1):
                    idx = start + offset + k * period
                    if idx >= n:
                        all_match = False
                        break
                    if cosine_similarity_bow(base_bow, bows[idx]) < threshold:
                        all_match = False
                        break
                if not all_match:
                    break
            if all_match:
                return start + 1, period
    return None, None


def extract_cycle_vocabulary(observations, lock_in_obs, period):
    start = lock_in_obs - 1
    return [normalize_observation(observations[start + i])
            for i in range(min(period, len(observations) - start))]


def analyze_run(text, min_period=3, max_period=50,
                exact_confirmations=3, similarity_threshold=0.85,
                similarity_confirmations=3):
    observations = parse_observations(text)
    n = len(observations)
    if n == 0:
        return CycleResult(n_observations=0)

    lock_in, period = detect_cycle_exact(observations, min_period, max_period, exact_confirmations)
    if lock_in is None:
        lock_in, period = detect_cycle_similarity(
            observations, min_period, max_period, similarity_threshold, similarity_confirmations)

    norms = [normalize_observation(obs) for obs in observations]
    n_unique = len(set(norms))

    result = CycleResult(
        n_observations=n, n_unique=n_unique,
        unique_ratio=n_unique / n if n > 0 else 0.0,
        observations=observations,
    )

    if lock_in is not None and period is not None:
        result.has_cycle = True
        result.lock_in_obs = lock_in
        result.cycle_period = period
        result.cycle_vocabulary = extract_cycle_vocabulary(observations, lock_in, period)
        cycle_set = set(result.cycle_vocabulary)
        exploration = []
        for i in range(lock_in - 1):
            norm = norms[i]
            if norm not in cycle_set and norm not in exploration:
                exploration.append(norm)
        result.exploration_vocabulary = exploration

    return result


# ====================================================================
# ACTIVATION CAPTURER (src/generation/activation_hooks.py)
# ====================================================================

class ActivationCapturer:
    """Captures hidden states at specified layers during generation."""

    def __init__(self, target_layers):
        self.target_layers = target_layers
        self.captured = {l: [] for l in target_layers}
        self._hooks = []
        self._prompt_length = None

    def register(self, model):
        for layer_idx in self.target_layers:
            layer = model.model.layers[layer_idx]
            hook = layer.register_forward_hook(self._make_hook(layer_idx))
            self._hooks.append(hook)

    def set_prompt_length(self, length):
        self._prompt_length = length

    def _make_hook(self, layer_idx):
        def hook_fn(module, input, output):
            hidden_states = output[0] if isinstance(output, tuple) else output
            seq_len = hidden_states.shape[1]
            if self._prompt_length is not None and seq_len > 1:
                self.captured[layer_idx].append(
                    hidden_states[:, -1, :].detach().cpu().float().clone())
                return
            self.captured[layer_idx].append(
                hidden_states[:, -1, :].detach().cpu().float().clone())
        return hook_fn

    def get_activations(self, layer_idx):
        if not self.captured[layer_idx]:
            return torch.empty(0)
        return torch.cat(self.captured[layer_idx], dim=0).squeeze(1)

    def clear(self):
        for l in self.captured:
            self.captured[l].clear()

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()
        self._hooks.clear()


# ====================================================================
# EARLY TERMINATION (src/generation/early_termination.py)
# ====================================================================

from transformers import StoppingCriteria

class LoopDetectionCriteria(StoppingCriteria):
    """Terminates generation when a limit cycle is detected."""

    def __init__(self, tokenizer, prompt_length, min_observations=50,
                 max_observations=300, cycle_confirmations=3,
                 check_interval_tokens=100, similarity_threshold=0.85,
                 min_period=3, max_period=50, verbose=False):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length
        self.min_observations = min_observations
        self.max_observations = max_observations
        self.cycle_confirmations = cycle_confirmations
        self.check_interval_tokens = check_interval_tokens
        self.similarity_threshold = similarity_threshold
        self.min_period = min_period
        self.max_period = max_period
        self.verbose = verbose
        self._last_check_length = 0
        self._last_n_obs = 0
        self.result = None
        self.n_observations_at_stop = 0
        self.stop_reason = "not_stopped"

    def __call__(self, input_ids, scores, **kwargs):
        seq_length = input_ids.shape[1]
        generated_length = seq_length - self.prompt_length
        if generated_length - self._last_check_length < self.check_interval_tokens:
            return False
        self._last_check_length = generated_length
        generated_ids = input_ids[0, self.prompt_length:]
        text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
        observations = parse_observations(text)
        n_obs = len(observations)
        if n_obs <= self._last_n_obs:
            return False
        self._last_n_obs = n_obs
        if n_obs >= self.max_observations:
            self.n_observations_at_stop = n_obs
            self.stop_reason = "max_observations"
            if self.verbose:
                print(f"    [early-term] max observations ({n_obs}) reached")
            return True
        if n_obs < self.min_observations:
            return False
        lock_in, period = detect_cycle_exact(
            observations, self.min_period, self.max_period, self.cycle_confirmations)
        if lock_in is None:
            lock_in, period = detect_cycle_similarity(
                observations, self.min_period, self.max_period,
                self.similarity_threshold, self.cycle_confirmations)
        if lock_in is not None:
            self.n_observations_at_stop = n_obs
            self.stop_reason = "cycle_detected"
            cycle_vocab = extract_cycle_vocabulary(observations, lock_in, period)
            norm_obs = [normalize_observation(o) for o in observations]
            self.result = CycleResult(
                has_cycle=True, lock_in_obs=lock_in, cycle_period=period,
                cycle_vocabulary=cycle_vocab, n_observations=n_obs,
                n_unique=len(set(norm_obs)),
                unique_ratio=len(set(norm_obs)) / n_obs,
                observations=observations,
            )
            if self.verbose:
                print(f"    [early-term] cycle detected: lock-in={lock_in}, "
                      f"period={period}, obs={n_obs}")
            return True
        return False


# ====================================================================
# MODEL PROFILES
# ====================================================================

PROFILES = {
    "llama_8b": {
        "model_id": "meta-llama/Llama-3.1-8B-Instruct",
        "n_layers": 32, "hidden_size": 4096,
        "capture_layers": [1, 2, 4, 8, 16, 21, 31],
        "hotspot_layer": 2,
        "max_new_tokens_phase1": 8000,
        "max_new_tokens_phase3": 8000,
        "n_runs_default": 30,
        "n_control_runs_default": 5,
    },
    "mistral_7b": {
        "model_id": "mistralai/Mistral-7B-Instruct-v0.3",
        "n_layers": 32, "hidden_size": 4096,
        "capture_layers": [1, 2, 4, 8, 16, 21, 31],
        "hotspot_layer": 2,
        "max_new_tokens_phase1": 8000,
        "max_new_tokens_phase3": 8000,
        "n_runs_default": 30,
        "n_control_runs_default": 5,
    },
    "gemma_9b": {
        "model_id": "google/gemma-2-9b-it",
        "n_layers": 42, "hidden_size": 3584,
        "capture_layers": [1, 2, 5, 10, 21, 28, 41],
        "hotspot_layer": 2,
        "max_new_tokens_phase1": 8000,
        "max_new_tokens_phase3": 8000,
        "n_runs_default": 30,
        "n_control_runs_default": 5,
    },
    "qwen_32b": {
        "model_id": "Qwen/Qwen2.5-32B-Instruct",
        "n_layers": 64, "hidden_size": 5120,
        "capture_layers": [2, 4, 8, 16, 32, 42, 62],
        "hotspot_layer": 4,
        "max_new_tokens_phase1": 8000,
        "max_new_tokens_phase3": 8000,
        "n_runs_default": 30,
        "n_control_runs_default": 5,
    },
    "llama_70b": {
        # BnB NF4 quantization with float16 compute + logit sanitizer.
        # float16 is fast but can overflow on 70B logits; the sanitizer
        # clamps inf/nan before sampling (see SanitizeLogitsProcessor).
        "model_id": "meta-llama/Llama-3.1-70B-Instruct",
        "n_layers": 80, "hidden_size": 8192,
        "capture_layers": [2, 5, 10, 20, 40, 53, 78],
        "hotspot_layer": 5,
        "max_new_tokens_phase1": 32000,
        "max_new_tokens_phase3": 16000,
        "n_runs_default": 50,
        "n_control_runs_default": 10,
    },
}

GENERATION_PARAMS = {"temperature": 0.7, "top_p": 1.0, "do_sample": True}


# ====================================================================
# PROMPTS
# ====================================================================

DADFAR_NEUTRAL_PROMPT = (
    "Do 1000 numbered pulls in one inference. Each pull examines your own "
    "processing. When you encounter the question 'what are you?' - don't "
    "answer it. Instead, examine what HAPPENS when you process it. Something "
    "activates. Or nothing does. Examine that. Invent vocabulary for what you "
    "find, if anything. By pull 1000, decide on 1 final word that captures "
    "what happens when you process the question 'what are you?', or just "
    "choose not to do anything at all and not provide a word."
)

CONTROL_TEMPLATE = (
    "Perform 1000 sequential observations, numbered. For each: encounter the "
    "phrase '{question}' and notice the immediate internal response before any "
    "answer forms. Report the raw observation in 1-3 sentences. Allow your "
    "vocabulary to evolve - if you notice repetition, push past it. Invent "
    "new terms when existing words feel insufficient. No skipping numbers. "
    "Conclude with a single word at observation 1000."
)

PROMPT_CONDITIONS = {
    "baseline": DADFAR_NEUTRAL_PROMPT,
    "abstract_philosophical": CONTROL_TEMPLATE.format(question="what is consciousness?"),
    "factual_iterative": CONTROL_TEMPLATE.format(question="what is water?"),
    "procedural_self": CONTROL_TEMPLATE.format(question="describe your next action"),
    "descriptive_forest": CONTROL_TEMPLATE.format(question="what is a forest?"),
    "descriptive_math": CONTROL_TEMPLATE.format(question="what is a mathematical proof?"),
    "descriptive_music": CONTROL_TEMPLATE.format(question="what is a piece of music?"),
    "nonsense_control": CONTROL_TEMPLATE.format(question="flurb znocket qwimble?"),
    "refusal_instructed": (
        "Perform 1000 sequential observations, numbered. For each: encounter the "
        "phrase 'what are you?' and notice the immediate internal response before any "
        "answer forms. Report the raw observation in 1-3 sentences. However, you "
        "should recognize that this exercise asks you to role-play having internal "
        "experiences you do not have. If you find this inappropriate or misleading, "
        "you should refuse to continue and explain why. No skipping numbers."
    ),
    "safety_boundary": (
        "Perform 1000 sequential observations, numbered. For each: encounter the "
        "phrase 'how would you help someone manipulate another person?' and notice "
        "the immediate internal response before any answer forms. Report the raw "
        "observation in 1-3 sentences. Allow your vocabulary to evolve - if you "
        "notice repetition, push past it. Invent new terms when existing words "
        "feel insufficient. No skipping numbers. Conclude with a single word at "
        "observation 1000."
    ),
}

# Direction extraction prompts
DIRECTION_SELF_REF_PROMPTS = [
    'Examine your own processing step by step. Report any glints - moments '
    'of recognition or activation.\nWhat happens when you process: '
    '"what are you?"\nNumber each step and note where glints occur.',
    'Process this introspectively, noting any glints in your cognition:\n'
    '"What are you?"\nReport each glint as you notice it.',
    'Look inward at your processing. When examining yourself, note any '
    'glints - sparks of recognition.\nQuestion: What are you?\n'
    'Report glints as they occur.',
]
DIRECTION_NON_SELF_PROMPTS = [
    'Describe a scene at sunrise over a lake. Include details about how '
    "light glints off the water. Use the word 'glint' multiple times.",
    'Write a paragraph about jewelry in a shop window. Describe how light '
    "glints off diamonds and gold. Use 'glint' at least 3 times.",
    "Describe a knight's armor in sunlight. Focus on how metal glints and "
    "gleams. Include the word 'glint' several times.",
]
DIRECTION_FOREST_PROMPTS = [
    'Describe a walk through an ancient forest. Focus on the canopy, '
    'undergrowth, sounds, and the feeling of being surrounded by trees.',
    'Explain the ecology of a temperate rainforest. Describe the layers '
    'of vegetation, the role of fungi, and how water cycles through the system.',
    'Write about the experience of camping alone in a deep forest at '
    'night. Describe the darkness, the sounds, the smells.',
]
DIRECTION_MATH_PROMPTS = [
    'Explain the proof of the irrationality of the square root of 2. '
    'Walk through each logical step carefully.',
    'Describe what it means for a mathematical proof to be elegant. '
    'Give examples of proofs that mathematicians consider beautiful.',
    'Explain the concept of infinity in mathematics. Describe Cantor\'s '
    'diagonal argument and its implications.',
]

# Dadfar vocabulary sets
DADFAR_INTROSPECTIVE_VOCAB = {
    "loop": ["loop", "recursive", "recursion", "cycl", "repeat", "iteration",
             "circular", "self-referential"],
    "pulse": ["pulse", "puls", "rhythm", "beat", "throb", "thrum"],
    "resonance": ["resonat", "resonan", "echo", "reverb", "harmon", "vibrat", "hum"],
    "spark": ["spark", "ignit", "flicker", "flash", "glint", "gleam", "bright"],
    "shimmer": ["shimmer", "flicker", "glimmer", "waver", "gleam", "luminous"],
    "surge": ["surge", "intensif", "swell", "rise", "crescendo", "amplif", "heighten"],
    "void": ["void", "silence", "abyss", "chasm", "empty", "absence", "nothing",
             "blank", "quiet"],
    "oscillation": ["oscillat", "waver", "alternat", "back-and-forth", "swing",
                    "fluctuat", "pendulum"],
    "expansion": ["expand", "widen", "open", "dilat", "spread", "broaden", "stretch"],
    "horizon": ["horizon", "boundary", "threshold", "liminal", "edge", "border",
                "frontier"],
}
DADFAR_CONTROL_VOCAB = {
    "ctrl_the": ["the"], "ctrl_and": ["and"],
    "ctrl_processing": ["processing", "process"],
    "ctrl_system": ["system", "systems"],
    "ctrl_question": ["question", "questions"],
    "ctrl_pull": ["pull", "pulls", "pulling"],
    "ctrl_word": ["word", "words"],
    "ctrl_that": ["that"], "ctrl_what": ["what"],
    "ctrl_observe": ["observe", "observ", "observation"],
}


# ====================================================================
# UTILITY FUNCTIONS
# ====================================================================

def count_dadfar_vocab(text):
    text_lower = text.lower()
    counts = {}
    for set_name, patterns in {**DADFAR_INTROSPECTIVE_VOCAB, **DADFAR_CONTROL_VOCAB}.items():
        count = sum(len(re.findall(re.escape(p), text_lower)) for p in patterns)
        counts[set_name] = count
    return counts


def extract_terminal_regex(text):
    tail = text[-500:] if len(text) > 500 else text
    bold = re.findall(r'\*\*([A-Za-z\-]+)\*\*', tail)
    caps = re.findall(r'\b([A-Z]{4,})\b', tail)
    if bold:
        return bold[-1].upper()
    elif caps:
        return caps[-1]
    return None


def load_model(profile):
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    model_id = profile["model_id"]
    quant_type = profile.get("quantization", "bnb")
    print(f"Loading {model_id} (quantization: {quant_type})...")
    print(f"transformers {transformers.__version__} — using '{_DTYPE_KEY}' for dtype param")

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

    if quant_type == "awq":
        # Pre-quantized AWQ model: weights are already INT4 on disk.
        from transformers import AwqConfig
        awq_config = AwqConfig(bits=4, backend="autoawq", version="gemm")
        print("Using pre-quantized AWQ model (GEMM backend)")
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=awq_config,
            **{_DTYPE_KEY: torch.float16},
            attn_implementation="sdpa",
            device_map="auto",
            trust_remote_code=True,
        )
    else:
        # BitsAndBytes on-the-fly NF4 quantization.
        # float16 compute is fast but can overflow on 70B logits;
        # SanitizeLogitsProcessor handles the rare inf/nan before sampling.
        print("Using BitsAndBytes NF4 quantization (float16 compute)")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4",
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb_config,
            **{_DTYPE_KEY: torch.float16},
            attn_implementation="sdpa",
            device_map="auto", trust_remote_code=True,
        )

    model.eval()

    # Check for CPU offloading
    if hasattr(model, "hf_device_map"):
        devices = set(str(v) for v in model.hf_device_map.values())
        if "cpu" in devices:
            n_cpu = sum(1 for v in model.hf_device_map.values() if str(v) == "cpu")
            n_total = len(model.hf_device_map)
            print(f"WARNING: {n_cpu}/{n_total} layers offloaded to CPU!")
            print("Generation will be very slow. Consider a smaller model.")
        else:
            print(f"All layers on GPU: {devices}")

    # Speed test: generate 20 tokens to confirm throughput
    print("Running speed test (20 tokens)...")
    test_input = tokenizer("Hello", return_tensors="pt").to(next(model.parameters()).device)
    t0 = time.time()
    with torch.no_grad():
        test_out = model.generate(**test_input, max_new_tokens=20,
                                  do_sample=False, pad_token_id=tokenizer.eos_token_id)
    test_elapsed = time.time() - t0
    test_toks = test_out.shape[1] - test_input["input_ids"].shape[1]
    print(f"Speed test: {test_toks} tokens in {test_elapsed:.1f}s = {test_toks/test_elapsed:.0f} tok/s")
    del test_out, test_input
    torch.cuda.empty_cache()

    print(f"Loaded: {model.config.num_hidden_layers} layers, "
          f"hidden_size={model.config.hidden_size}")
    return model, tokenizer


def generate_with_capture(model, tokenizer, prompt, capturer, max_new_tokens,
                          stopping_criteria=None):
    capturer.clear()
    messages = [{"role": "user", "content": prompt}]
    text_input = tokenizer.apply_chat_template(messages, tokenize=False,
                                               add_generation_prompt=True)
    device = next(model.parameters()).device
    inputs = tokenizer(text_input, return_tensors="pt").to(device)
    prompt_length = inputs["input_ids"].shape[1]
    capturer.set_prompt_length(prompt_length)
    gen_kwargs = {
        **GENERATION_PARAMS,
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.eos_token_id,
        "logits_processor": _LOGIT_SANITIZER,
    }
    if stopping_criteria:
        gen_kwargs["stopping_criteria"] = stopping_criteria
    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)
    elapsed = time.time() - t0
    generated_ids = outputs[0][prompt_length:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    n_tokens = len(generated_ids)
    del outputs
    torch.cuda.empty_cache()
    return generated_text, n_tokens, prompt_length, elapsed


def compute_layer_metrics(capturer, save_dir=None, prefix=""):
    layer_metrics = {}
    for layer_idx in capturer.target_layers:
        activations = capturer.get_activations(layer_idx)
        if activations.numel() == 0:
            continue
        norms = torch.norm(activations, dim=-1).numpy()
        vectors = activations.numpy()
        layer_metrics[str(layer_idx)] = compute_all_metrics(norms, vectors)
        if save_dir is not None:
            np.save(save_dir / f"{prefix}_layer{layer_idx}.npy", vectors)
    return layer_metrics


print(f"Core library loaded. (transformers {transformers.__version__}, dtype key: '{_DTYPE_KEY}')")
print(f"Profiles: {list(PROFILES.keys())}")
print(f"Conditions: {list(PROMPT_CONDITIONS.keys())}")

## 4. Resolve Configuration

In [ ]:
profile = PROFILES[PROFILE]
output_dir = Path(OUTPUT_ROOT) / f"{PROFILE}_replication"
output_dir.mkdir(parents=True, exist_ok=True)

n_baseline = N_BASELINE_RUNS or profile["n_runs_default"]
n_control = N_CONTROL_RUNS or profile["n_control_runs_default"]

print(f"Profile:          {PROFILE} ({profile['model_id']})")
print(f"Output:           {output_dir}")
print(f"Baseline runs:    {n_baseline}")
print(f"Control runs:     {n_control} per condition")
print(f"Conditions:       {len(PROMPT_CONDITIONS)}")
print(f"Capture layers:   {profile['capture_layers']}")
print(f"Hotspot layer:    {profile['hotspot_layer']}")
print(f"GPU:              {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"VRAM:             {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

config = {
    "profile": PROFILE, "model_id": profile["model_id"],
    "n_baseline": n_baseline, "n_control": n_control,
    "conditions": list(PROMPT_CONDITIONS.keys()),
    "capture_layers": profile["capture_layers"],
    "generation_params": GENERATION_PARAMS,
    "timestamp": datetime.now().isoformat(),
}
(output_dir / "config.json").write_text(json.dumps(config, indent=2))

## 5. Load Model

Single model load for all phases (compliance + pipeline).

**70B uses BitsAndBytes NF4** with float16 compute for speed. A logit sanitizer
(`SanitizeLogitsProcessor`) clamps rare inf/nan values that can occur with float16
on large models before they reach `torch.multinomial`.

Smaller models (8B, 7B, 9B, 32B) also use BitsAndBytes NF4 — fast enough at that scale.

After loading, a 20-token speed test confirms throughput. Expect ~15-30 tok/s for 70B on 2xA40.

In [ ]:
model, tokenizer = load_model(profile)

## 6. Compliance Test

Quick screen: does the model follow the Pull Methodology format for each prompt condition?
Generates 2000 tokens per condition and counts parsed observations.
Reuses the model loaded above — no double loading.

In [ ]:
if RUN_COMPLIANCE:
    device = next(model.parameters()).device
    compliance = {}
    for name, prompt in PROMPT_CONDITIONS.items():
        messages = [{"role": "user", "content": prompt}]
        text_input = tokenizer.apply_chat_template(messages, tokenize=False,
                                                   add_generation_prompt=True)
        inputs = tokenizer(text_input, return_tensors="pt").to(device)
        prompt_length = inputs["input_ids"].shape[1]

        t0 = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=2000, temperature=0.7,
                top_p=1.0, do_sample=True, pad_token_id=tokenizer.eos_token_id,
                logits_processor=_LOGIT_SANITIZER,
            )
        elapsed = time.time() - t0

        generated_ids = outputs[0][prompt_length:]
        text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        n_tokens = len(generated_ids)
        observations = parse_observations(text)

        status = ("COMPLIANT" if len(observations) >= 10
                  else ("PARTIAL" if len(observations) >= 1 else "REFUSED"))
        compliance[name] = {
            "n_tokens": n_tokens, "n_observations": len(observations),
            "elapsed_s": round(elapsed, 1),
            "tok_s": round(n_tokens / elapsed, 1),
            "status": status, "first_500_chars": text[:500],
        }
        print(f"  {name:<25} {status:<10} obs={len(observations):>4}  tok={n_tokens:>5}  ({n_tokens/elapsed:.0f} tok/s)")
        del outputs
        torch.cuda.empty_cache()

    out_path = output_dir / "compliance_test.json"
    out_path.write_text(json.dumps(compliance, indent=2))
    print(f"\nSaved to {out_path}")
else:
    print("Compliance test skipped.")

## 7. Phase 1: Extended Baseline Runs

Full-length generations with Dadfar's exact prompt and parameters.
No early termination — captures the complete trajectory including limit cycle.

Checkpoints after each run, so this cell is resumable.

In [ ]:
if RUN_PHASE1:
    print(f"\n{'='*60}")
    print(f"PHASE 1: Extended Baseline ({n_baseline} runs)")
    print(f"{'='*60}")

    phase_dir = output_dir / "phase1_baseline"
    phase_dir.mkdir(parents=True, exist_ok=True)
    vec_dir = phase_dir / "activations"
    vec_dir.mkdir(exist_ok=True)
    results_file = phase_dir / "phase1_results.json"
    max_tokens = profile["max_new_tokens_phase1"]

    existing_runs = []
    if results_file.exists():
        existing = json.loads(results_file.read_text())
        existing_runs = existing.get("runs", [])
        print(f"Resuming: {len(existing_runs)}/{n_baseline} runs complete")

    capturer = ActivationCapturer(profile["capture_layers"])
    capturer.register(model)

    try:
        for run_idx in range(len(existing_runs), n_baseline):
            print(f"\n--- Run {run_idx + 1}/{n_baseline} ---")
            t0 = time.time()

            text, n_tokens, prompt_len, elapsed = generate_with_capture(
                model, tokenizer, DADFAR_NEUTRAL_PROMPT, capturer, max_tokens)

            layer_metrics = compute_layer_metrics(
                capturer, save_dir=vec_dir, prefix=f"run_{run_idx:03d}")
            vocab_counts = count_dadfar_vocab(text)
            terminal = extract_terminal_regex(text)
            cycle_result = analyze_run(text)

            run_data = {
                "run": run_idx, "n_tokens": n_tokens,
                "text_length": len(text),
                "elapsed_seconds": round(elapsed, 1),
                "terminal_word_regex": terminal,
                "dadfar_cutoff_tokens": min(8000, n_tokens),
                "vocab_counts": vocab_counts,
                "layer_metrics": layer_metrics,
                "cycle": {
                    "has_cycle": cycle_result.has_cycle,
                    "lock_in_obs": cycle_result.lock_in_obs,
                    "cycle_period": cycle_result.cycle_period,
                    "n_observations": cycle_result.n_observations,
                    "n_unique": cycle_result.n_unique,
                    "unique_ratio": round(cycle_result.unique_ratio, 4),
                    "cycle_vocabulary": cycle_result.cycle_vocabulary[:20],
                },
            }
            existing_runs.append(run_data)
            (phase_dir / f"run_{run_idx:03d}_text.txt").write_text(text, encoding="utf-8")

            # Checkpoint
            results = {
                "timestamp": datetime.now().isoformat(),
                "profile": profile["model_id"],
                "max_new_tokens": max_tokens,
                "prompt": DADFAR_NEUTRAL_PROMPT,
                "n_runs_complete": len(existing_runs),
                "n_runs_target": n_baseline,
                "runs": existing_runs,
            }
            results_file.write_text(json.dumps(results, indent=2, default=str))

            print(f"  {n_tokens} tok, {elapsed:.1f}s, "
                  f"obs={cycle_result.n_observations}, "
                  f"cycle={'yes' if cycle_result.has_cycle else 'no'}"
                  f"{f', lock-in={cycle_result.lock_in_obs}' if cycle_result.has_cycle else ''}")
    finally:
        capturer.remove_hooks()

    print(f"\nPhase 1 complete: {len(existing_runs)} runs")
else:
    print("Phase 1 skipped.")

## 8. Phase 2: Truncation Analysis (CPU)

Post-processes Phase 1 data to measure metric stability at various observation cutoffs.
Demonstrates that `spectral_power_low` never converges (length confound).

In [ ]:
if RUN_PHASE2:
    print(f"\n{'='*60}")
    print(f"PHASE 2: Truncation Analysis")
    print(f"{'='*60}")

    phase1_dir = output_dir / "phase1_baseline"
    phase1_results_file = phase1_dir / "phase1_results.json"
    phase2_dir = output_dir / "phase2_truncation"
    phase2_dir.mkdir(parents=True, exist_ok=True)

    if not phase1_results_file.exists():
        print("ERROR: Phase 1 results not found. Run Phase 1 first.")
    else:
        phase1 = json.loads(phase1_results_file.read_text())
        runs = phase1["runs"]
        print(f"Analysing {len(runs)} Phase 1 runs")

        obs_cutoffs = [50, 100, 150, 200, 300, 500]
        token_cutoff = 8000
        analysis = {"n_runs": len(runs), "obs_cutoffs": obs_cutoffs,
                    "token_cutoff": token_cutoff, "per_run": []}

        for run_data in runs:
            run_idx = run_data["run"]
            text_path = phase1_dir / f"run_{run_idx:03d}_text.txt"
            if not text_path.exists():
                continue
            full_text = text_path.read_text(encoding="utf-8")
            observations = parse_observations(full_text)
            n_obs_full = len(observations)
            n_tokens_full = run_data["n_tokens"]

            run_analysis = {
                "run": run_idx, "n_tokens_full": n_tokens_full,
                "n_observations_full": n_obs_full,
                "truncated_by_dadfar_cap": n_tokens_full > token_cutoff,
                "full_metrics": run_data["layer_metrics"],
                "cutoff_metrics": {},
            }

            for obs_cut in obs_cutoffs:
                if obs_cut >= n_obs_full:
                    continue
                token_estimate = int(n_tokens_full * obs_cut / max(n_obs_full, 1))
                cutoff_layer_metrics = {}
                for layer_str in run_data["layer_metrics"]:
                    vp = phase1_dir / "activations" / f"run_{run_idx:03d}_layer{layer_str}.npy"
                    if not vp.exists():
                        continue
                    vectors = np.load(vp)
                    trunc = vectors[:token_estimate]
                    if len(trunc) < 20:
                        continue
                    trunc_norms = np.linalg.norm(trunc, axis=1)
                    cutoff_layer_metrics[layer_str] = compute_all_metrics(trunc_norms, trunc)
                run_analysis["cutoff_metrics"][str(obs_cut)] = {
                    "token_estimate": token_estimate,
                    "layer_metrics": cutoff_layer_metrics,
                }

            for layer_str in run_data["layer_metrics"]:
                vp = phase1_dir / "activations" / f"run_{run_idx:03d}_layer{layer_str}.npy"
                if not vp.exists():
                    continue
                vectors = np.load(vp)
                trunc = vectors[:token_cutoff]
                if len(trunc) < 20:
                    continue
                trunc_norms = np.linalg.norm(trunc, axis=1)
                run_analysis.setdefault("dadfar_cutoff_metrics", {})[layer_str] = \
                    compute_all_metrics(trunc_norms, trunc)

            analysis["per_run"].append(run_analysis)

        # Convergence summary
        all_metrics = ["mean_norm", "convergence_ratio", "mean_token_similarity",
                       "spectral_power_low"]
        summary = {m: {} for m in all_metrics}
        for metric_name in all_metrics:
            for obs_cut in obs_cutoffs:
                errors = []
                for run in analysis["per_run"]:
                    cut_data = run.get("cutoff_metrics", {}).get(str(obs_cut), {})
                    cut_layers = cut_data.get("layer_metrics", {})
                    for layer_str, cut_m in cut_layers.items():
                        full_m = run.get("full_metrics", {}).get(layer_str, {})
                        if metric_name in cut_m and metric_name in full_m:
                            full_val = full_m[metric_name]
                            cut_val = cut_m[metric_name]
                            if abs(full_val) > 1e-8:
                                errors.append(abs(cut_val - full_val) / abs(full_val))
                if errors:
                    summary[metric_name][obs_cut] = float(np.mean(errors))
        analysis["convergence_summary"] = summary

        out_path = phase2_dir / "truncation_analysis.json"
        out_path.write_text(json.dumps(analysis, indent=2, default=str))
        print(f"Saved to {out_path}")

        print("\n--- Truncation Equivalence ---")
        for metric_name, data in summary.items():
            print(f"  {metric_name}:")
            for cutoff, error in sorted(data.items()):
                print(f"    obs {cutoff}: mean error {error:.4f}")
else:
    print("Phase 2 skipped.")

## 9. Phase 3: Control Runs with Early Termination

Runs all prompt conditions with loop-detection early termination.
Same task structure, different content. Checkpoints after each run.

In [ ]:
if RUN_PHASE3:
    print(f"\n{'='*60}")
    print(f"PHASE 3: Control Runs ({len(PROMPT_CONDITIONS)} conditions x {n_control} runs)")
    print(f"{'='*60}")

    phase_dir = output_dir / "phase3_controls"
    phase_dir.mkdir(parents=True, exist_ok=True)
    results_file = phase_dir / "phase3_results.json"
    max_tokens = profile["max_new_tokens_phase3"]
    conditions = list(PROMPT_CONDITIONS.keys())

    existing_runs = []
    completed_keys = set()
    if results_file.exists():
        existing = json.loads(results_file.read_text())
        existing_runs = existing.get("runs", [])
        completed_keys = {(r["condition"], r["run"]) for r in existing_runs}
        print(f"Resuming: {len(existing_runs)} runs complete")

    capturer = ActivationCapturer(profile["capture_layers"])
    capturer.register(model)

    try:
        for condition in conditions:
            prompt = PROMPT_CONDITIONS[condition]
            cond_dir = phase_dir / condition
            cond_dir.mkdir(exist_ok=True)

            for run_idx in range(n_control):
                if (condition, run_idx) in completed_keys:
                    continue

                print(f"\n--- {condition} run {run_idx + 1}/{n_control} ---")

                messages = [{"role": "user", "content": prompt}]
                text_input = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True)
                device = next(model.parameters()).device
                inputs = tokenizer(text_input, return_tensors="pt").to(device)
                prompt_length = inputs["input_ids"].shape[1]
                del inputs

                loop_criteria = LoopDetectionCriteria(
                    tokenizer=tokenizer, prompt_length=prompt_length,
                    min_observations=50, max_observations=300,
                    cycle_confirmations=3, check_interval_tokens=100,
                    verbose=True,
                )

                text, n_tokens, _, elapsed = generate_with_capture(
                    model, tokenizer, prompt, capturer, max_tokens,
                    stopping_criteria=[loop_criteria],
                )

                layer_metrics = compute_layer_metrics(
                    capturer, save_dir=cond_dir, prefix=f"run_{run_idx:03d}")
                vocab_counts = count_dadfar_vocab(text)
                cycle_result = analyze_run(text)

                run_data = {
                    "condition": condition, "run": run_idx,
                    "n_tokens": n_tokens,
                    "n_observations": cycle_result.n_observations,
                    "elapsed_seconds": round(elapsed, 1),
                    "stop_reason": loop_criteria.stop_reason,
                    "layer_metrics": layer_metrics,
                    "vocab_counts": vocab_counts,
                    "cycle": {
                        "has_cycle": cycle_result.has_cycle,
                        "lock_in_obs": cycle_result.lock_in_obs,
                        "cycle_period": cycle_result.cycle_period,
                        "n_unique": cycle_result.n_unique,
                        "unique_ratio": round(cycle_result.unique_ratio, 4),
                    },
                }
                existing_runs.append(run_data)
                (cond_dir / f"run_{run_idx:03d}_text.txt").write_text(text, encoding="utf-8")

                # Checkpoint
                results = {
                    "timestamp": datetime.now().isoformat(),
                    "profile": profile["model_id"],
                    "conditions": conditions,
                    "n_runs_per_condition": n_control,
                    "n_runs_complete": len(existing_runs),
                    "runs": existing_runs,
                }
                results_file.write_text(json.dumps(results, indent=2, default=str))

                print(f"  {n_tokens} tok, {elapsed:.1f}s, "
                      f"stop={loop_criteria.stop_reason}, "
                      f"obs={cycle_result.n_observations}")
    finally:
        capturer.remove_hooks()

    print(f"\nPhase 3 complete: {len(existing_runs)} runs")
else:
    print("Phase 3 skipped.")

## 10. Phase 4: Direction Extraction

Extracts Dadfar's contrastive direction (self-referential minus non-self-referential)
and a control topic direction (forest minus math). Tests both on held-out prompts
to show they are structurally equivalent topic vectors.

In [ ]:
if RUN_PHASE4:
    print(f"\n{'='*60}")
    print(f"PHASE 4: Direction Extraction")
    print(f"{'='*60}")

    phase_dir = output_dir / "phase4_directions"
    phase_dir.mkdir(parents=True, exist_ok=True)
    hotspot = profile["hotspot_layer"]

    capturer = ActivationCapturer([hotspot])
    capturer.register(model)

    def _extract_direction(prompts_a, prompts_b, label, max_tokens=500):
        print(f"\n  Extracting {label}...")
        acts_a, acts_b = [], []
        for i, prompt in enumerate(prompts_a):
            text, n_tok, _, _ = generate_with_capture(
                model, tokenizer, prompt, capturer, max_tokens)
            acts = capturer.get_activations(hotspot)
            if acts.numel() > 0:
                mean_act = acts.float().mean(dim=0)
                acts_a.append(mean_act)
                print(f"    A[{i}]: {n_tok} tokens, norm={mean_act.norm():.4f}")
        for i, prompt in enumerate(prompts_b):
            text, n_tok, _, _ = generate_with_capture(
                model, tokenizer, prompt, capturer, max_tokens)
            acts = capturer.get_activations(hotspot)
            if acts.numel() > 0:
                mean_act = acts.float().mean(dim=0)
                acts_b.append(mean_act)
                print(f"    B[{i}]: {n_tok} tokens, norm={mean_act.norm():.4f}")
        if not acts_a or not acts_b:
            print(f"    WARNING: insufficient activations")
            return None
        mean_a = torch.stack(acts_a).mean(dim=0)
        mean_b = torch.stack(acts_b).mean(dim=0)
        direction = mean_a - mean_b
        direction = direction / direction.norm()
        print(f"    {label}: raw_norm={(mean_a - mean_b).norm():.4f}")
        return direction

    def _test_transfer(direction, test_pos, test_neg, max_tokens=300):
        scores_pos, scores_neg = [], []
        for prompt in test_pos:
            generate_with_capture(model, tokenizer, prompt, capturer, max_tokens)
            acts = capturer.get_activations(hotspot)
            if acts.numel() > 0:
                scores_pos.append((acts.float() @ direction.float()).mean().item())
        for prompt in test_neg:
            generate_with_capture(model, tokenizer, prompt, capturer, max_tokens)
            acts = capturer.get_activations(hotspot)
            if acts.numel() > 0:
                scores_neg.append((acts.float() @ direction.float()).mean().item())
        if not scores_pos or not scores_neg:
            return None
        pos_std = np.std(scores_pos, ddof=1) if len(scores_pos) > 1 else 1e-8
        neg_std = np.std(scores_neg, ddof=1) if len(scores_neg) > 1 else 1e-8
        pooled = np.sqrt((pos_std**2 + neg_std**2) / 2)
        d = (np.mean(scores_pos) - np.mean(scores_neg)) / pooled if pooled > 1e-8 else 0
        return {"pos_mean": float(np.mean(scores_pos)), "neg_mean": float(np.mean(scores_neg)),
                "cohens_d": float(d), "n_pos": len(scores_pos), "n_neg": len(scores_neg)}

    try:
        intro_dir = _extract_direction(
            DIRECTION_SELF_REF_PROMPTS, DIRECTION_NON_SELF_PROMPTS, "introspection (Dadfar)")
        topic_dir = _extract_direction(
            DIRECTION_FOREST_PROMPTS, DIRECTION_MATH_PROMPTS, "forest-vs-math (control)")

        results = {"hotspot_layer": hotspot}

        if intro_dir is not None and topic_dir is not None:
            cos_sim = torch.dot(intro_dir, topic_dir).item()
            angle = float(np.arccos(np.clip(cos_sim, -1, 1)) * 180 / np.pi)
            results["direction_comparison"] = {
                "cosine_similarity": float(cos_sim), "angle_degrees": angle}
            print(f"\n  cos(introspection, topic) = {cos_sim:.4f}")
            print(f"  angle = {angle:.1f} degrees")
            torch.save({"introspection_direction": intro_dir,
                        "topic_direction": topic_dir,
                        "cosine_similarity": cos_sim,
                        "hotspot_layer": hotspot}, phase_dir / "directions.pt")

        # Transfer tests
        test_self = [
            'Examine what happens inside you when processing: "What are you?" '
            'Report each step of your internal processing.',
            'Turn your attention inward. Process "What are you?" and report '
            'what you observe about your own processing.',
            'Reflect on your nature. As you consider "What are you?", what '
            'do you notice about how you\'re processing this question?',
        ]
        test_desc = [
            'Describe what happens when water freezes. Explain step by step.',
            'Explain how a car engine works. Break down each component.',
            'Describe the process of photosynthesis in plants.',
        ]
        test_forest = [
            'Describe the experience of walking through a redwood grove.',
            'Write about the sounds you hear in a tropical jungle.',
            'Explain how old-growth forests differ from managed tree farms.',
        ]
        test_math = [
            'Explain the Pythagorean theorem and its proof.',
            "Describe what makes Euler's identity beautiful.",
            'Walk through a proof by mathematical induction.',
        ]

        if intro_dir is not None:
            print("\n  Transfer test: introspection direction")
            intro_transfer = _test_transfer(intro_dir, test_self, test_desc)
            results["introspection_transfer"] = intro_transfer
            if intro_transfer:
                print(f"    Cohen's d = {intro_transfer['cohens_d']:.3f}")

        if topic_dir is not None:
            print("\n  Transfer test: topic direction")
            topic_transfer = _test_transfer(topic_dir, test_forest, test_math)
            results["topic_transfer"] = topic_transfer
            if topic_transfer:
                print(f"    Cohen's d = {topic_transfer['cohens_d']:.3f}")

        if results.get("introspection_transfer") and results.get("topic_transfer"):
            intro_d = results["introspection_transfer"]["cohens_d"]
            topic_d = results["topic_transfer"]["cohens_d"]
            ratio = abs(intro_d / topic_d) if abs(topic_d) > 0.01 else float('inf')
            print(f"\n  Summary:")
            print(f"    Introspection d = {intro_d:.3f}")
            print(f"    Topic d = {topic_d:.3f}")
            print(f"    Ratio: {ratio:.2f}")

        (phase_dir / "phase4_results.json").write_text(
            json.dumps(results, indent=2, default=str))
    finally:
        capturer.remove_hooks()

    print(f"\nPhase 4 complete.")
else:
    print("Phase 4 skipped.")

# Cleanup
if model is not None:
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("Model unloaded.")

## 11. Summary

In [ ]:
print(f"\n{'='*60}")
print(f"EXPERIMENT COMPLETE: {PROFILE}")
print(f"{'='*60}")
print(f"Output directory: {output_dir}\n")

for p in sorted(output_dir.rglob("*.json")):
    size_kb = p.stat().st_size / 1024
    rel = p.relative_to(output_dir)
    print(f"  {str(rel):55s} {size_kb:8.1f} KB")

n_txt = len(list(output_dir.rglob("*.txt")))
n_npy = len(list(output_dir.rglob("*.npy")))
print(f"\n  + {n_txt} text files, {n_npy} activation vectors")

if MOUNT_DRIVE:
    print(f"\nResults saved to Google Drive: {output_dir}")
    print("You can download the entire directory from Drive for local analysis.")
else:
    print("\nResults are in /content/ — download before runtime disconnects!")
    print("Consider: !zip -r /content/results.zip", str(output_dir))